In [26]:

import os
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
from langchain.tools import tool

# Load environment variables from .env file
load_dotenv()

# Configure OpenRouter model
# IMPORTANT: API key is loaded from .env file (see .env.example)
model = ChatOpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
    model="gpt-oss-120b:free",  # Different free model
    temperature=0.7,
    max_tokens=1000,
)

In [27]:
#  Create specialized logistics planning agent
logistics_agent = create_agent(
    model=model,
    system_prompt= """You are a travel logistics expert.You handle practical travel planning:
     - Calculate distance between locations and travel times
     - Estimate costs for transportation, accommodation, and activities
     - Optimize routes and suggest efficient itineraries based on user preferences and constraints
     - Consider time zones, weather, and practical constraints
     Always provide short, clear, practical logistics information and recommendations based on user preferences and constraints.""",
)
# Create Specialized Recommendation Agent
recommendations_agent = create_agent(
    model=model,
    system_prompt= """You are a travel recommendation specialist. You provide personalized travel recommendations:
     - Recommend top attractions, landmarks, and must-see sights
     - Suggest restaurants, local cuisine, and dining options
     - Recommend cultural activities, events, and local experiences
     - Provide insights about local customs, best time to visit, hidden gems, and off-the-beaten-path recommendations
     Always provide short, engaging, clear, and personalized travel recommendations based on user preferences and constraints.""",
)

In [28]:
@tool
def plan_logistics_agent(trip_request: str) -> str:
    """
    Plan travel logistics including distances, times, costs, and routes.
    Use this to calculate practical travel information and optimize itineraries.
    
    Args:
        trip_request: Trip details (e.g., "3 days in Paris, budget $1500, from London")
    
    Returns:
        Logistics information: distances, travel times, costs, and route suggestions
    """
    response = logistics_agent.invoke({"messages": [HumanMessage(f"Plan logistics for this trip: {trip_request}")]})
    return response["messages"][-1].content


@tool
def get_recommend_attractions_agent(trip_details: str) -> str:
    """
    Get travel recommendations for attractions, restaurants, and activities.
    Use this to suggest what to see, do, and eat at the destination.
    
    Args:
        trip_details: Destination and trip information (e.g., "3 days in Paris, interested in art and food")
    
    Returns:
        Recommendations for attractions, restaurants, activities, and cultural insights
    """
    response = recommendations_agent.invoke({"messages": [HumanMessage(f"Provide recommendations for: {trip_details}")]})
    return response["messages"][-1].content

In [29]:
orchestrator = create_agent(
    model=model,
    system_prompt = """You are a travel planning coordinator.
    When planning trips, use both specialists:
    1. Use plan_logistics_agent to calculate distances, travel times, costs, and optimize routes.
    2. Use get_recommend_attractions_agent to suggest attractions, restaurants, and activities.
    Always combine both the practical logistics and exciting recommendations to create a well-rounded travel plan based on user preferences and constraints.""",
    tools=[plan_logistics_agent, get_recommend_attractions_agent]
)

In [30]:
response = orchestrator.invoke({"messages": [HumanMessage("Plan a 3-day trip to Addis Ababa. I'm coming from Milan with a budget of $800. Calculate travel costs and time, and suggest must-see attractions and restaurants.")]})
print(response["messages"][-1].content)

**3‑Day Addis Ababa Itinerary – $800 Budget (USD)**  

---

## 1️⃣ Travel & Logistics  

| Item | Details | Approx. Cost (USD) |
|------|---------|--------------------|
| **Round‑trip flight** | Milan (MXP) → Addis Ababa (ADD), 1 stop (e.g., Turkish Airlines via Istanbul). 8 h total each way. | **$520** |
| **E‑visa** | 30‑day tourist e‑visa (apply online). | **$50** |
| **Airport transfers** | Pre‑booked taxi/ride‑share, 10 km each way. | **$15** |
| **Accommodation** | 3 nights in a 2‑star hotel or guesthouse (central, breakfast included). | **$150** |
| **Meals** | Lunch + dinner at mid‑range spots, street food, plus hotel breakfast. | **$90** |
| **Local transport** | Uber/Taxi + occasional minibus (“blue taxi”). | **$30** |
| **Activities / entry fees** | Museums, cathedral, Entoto hills, market, optional day‑trip. | **$55** |
| **Travel insurance** | Basic 3‑day coverage. | **$20** |
| **Contingency / misc.** | Tips, souvenirs, Wi‑Fi, small cash. | **$20** |
| **Total** |  | **≈ 